In [ ]:
from glob import glob
from datasets import Dataset
import pickle
from umap import UMAP
from hdbscan import HDBSCAN
from bertopic import BERTopic
from nltk.corpus import stopwords
from bertopic.representation import MaximalMarginalRelevance, PartOfSpeech, LangChain
from bertopic.vectorizers import  ClassTfidfTransformer
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from collections import Counter

In [ ]:
import plotly.io as pio
pio.renderers.default = "vscode"

In [ ]:
save_path =  "val_data"# "books_modeling"
data_path = "val_data.csv"
embeds_path = "val_data.pkl"

In [ ]:
stoplist = list(set(stopwords.words('english'))) 

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("sentence-transformers/distiluse-base-multilingual-cased-v1", model_kwargs={"torch_dtype": "float16"})

In [ ]:
df = pd.read_csv(data_path)
texts = df["text"]

In [ ]:
embeddings = embedding_model.encode(list(texts), show_progress_bar=True)

In [ ]:
params = {  
            # TFIDF
            "reduce_frequent_words": True, "bm25_weighting": True,   
            "seed_words": [],
            "seed_multiplier": 4,
            # UMAP
            "n_neighbors": 20, "n_components": 5, "min_dist": 0.0, "metric_umap": "cosine", "random_state": 42,
            # HDBSCAN (change min_cluster_size for more/less topics?, default is 10, recommended to only increase above 10)
            "min_cluster_size": 8, "metric_hbd": "euclidean", "cluster_selection_method": "eom", "prediction_data": True,
            # Vectorizer model
            "stop_words": stoplist, "min_df": 2, "ngram_range": (1,4),
            # Representation models
            "diversity": 0.7
         }

In [ ]:
ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=params["reduce_frequent_words"], bm25_weighting=params["bm25_weighting"], 
                                     seed_words=params['seed_words'], seed_multiplier=params["seed_multiplier"])



In [ ]:

umap_model = UMAP(n_neighbors=params["n_neighbors"], 
                  n_components=params["n_components"], 
                  min_dist=params["min_dist"], 
                  metric=params["metric_umap"], 
                  random_state=params["random_state"])



In [ ]:
hdbscan_model = HDBSCAN(min_cluster_size=params["min_cluster_size"],
                        metric=params["metric_hbd"], 
                        cluster_selection_method=params["cluster_selection_method"], 
                        prediction_data=params["prediction_data"])

In [ ]:
vectorizer_model = CountVectorizer(stop_words=params["stop_words"], 
                                   min_df=params["min_df"], 
                                   ngram_range=params["ngram_range"])



In [ ]:

representation_models = [
    
                            MaximalMarginalRelevance(diversity=params["diversity"]),
                            PartOfSpeech("en_core_web_sm"),
                            # LangChainRepresentation(llm)
                            
                        ]

In [ ]:
topic_model = BERTopic(

    # Pipeline models
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    representation_model=representation_models,
    top_n_words=10,
    verbose=True,
    ctfidf_model=ctfidf_model,
    # nr_topics="auto",
    calculate_probabilities=True,
)

# Train model
topics, probs = topic_model.fit_transform(texts, embeddings)

In [ ]:
id2label = {0: 'HIGH', 1: 'MEDIUM', 2: 'LOW'}
labels = [id2label[x] for x in list(df["label"])]

In [ ]:
topics_per_class = topic_model.topics_per_class(texts, classes=labels)

In [ ]:
topic_model.visualize_topics()

In [ ]:
topic_model.visualize_heatmap()

In [ ]:
topic_model.visualize_topics_per_class(topics_per_class)